# 01 — `@dataclass`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- générer automatiquement `__init__`, `__repr__`, `__eq__` avec `@dataclass`
- utiliser `field(default_factory=...)` pour les valeurs par défaut mutables
- activer `slots=True`, `frozen=True`, `kw_only=True`
- post-traiter via `__post_init__`
- connaître les limites (pas de validation poussée → voir Pydantic)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, héritage, surcharge d'opérateurs
- type hints modernes
- `Protocol`, `TypedDict`

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- Pydantic v2 (notebook suivant)
- décorateurs custom (jour 3)

## Plan

1. Pourquoi `@dataclass`
2. La forme minimale
3. Valeurs par défaut et `field`
4. `default_factory` pour listes/dicts
5. `slots=True` : gain mémoire
6. `frozen=True` : immuabilité
7. `kw_only=True` : arguments nommés
8. `__post_init__` : validation légère
9. Héritage de dataclasses
10. Conversion en dict avec `asdict`
11. Limites et quand passer à Pydantic
12. Synthèse
13. Exercices

---

## 1. Pourquoi `@dataclass`

Écrire à la main `__init__`, `__repr__`, `__eq__` est fastidieux et source d'erreurs. `@dataclass` **génère** ces méthodes à partir des annotations de la classe.

In [ ]:
# Sans dataclass — 15 lignes
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    def __repr__(self) -> str:
        return f'Point(x={self.x}, y={self.y})'
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)


---

## 2. La forme minimale

Le décorateur, et des annotations. C'est tout.

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: float
    y: float


In [ ]:
p = Point(3.0, 4.0)


In [ ]:
p


In [ ]:
Point(3.0, 4.0) == Point(3.0, 4.0)


---

## 3. Valeurs par défaut et `field`

Les annotations peuvent avoir une valeur par défaut.

In [ ]:
@dataclass
class Salle:
    nom: str
    capacite: int = 10
    active: bool = True


In [ ]:
Salle('Mars')


In [ ]:
Salle('Venus', capacite=6)


---

## 4. `default_factory` pour listes/dicts

Interdiction stricte de mettre `= []` en défaut d'annotation : ça déclenche une `ValueError` volontaire. On utilise `field(default_factory=list)`.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Panier:
    articles: list[str] = field(default_factory=list)
    meta: dict[str, str] = field(default_factory=dict)


In [ ]:
Panier()


In [ ]:
p1 = Panier()
p2 = Panier()
p1.articles.append('pain')
p2.articles  # vide — pas de partage


---

## 5. `slots=True` : gain mémoire

Activer `slots=True` supprime `__dict__` et économise de la mémoire (10 à 30 %). Bonus : plus rapide, et empêche de créer des attributs non déclarés.

In [ ]:
@dataclass(slots=True)
class Coord:
    x: float
    y: float


In [ ]:
c = Coord(1.0, 2.0)
try:
    c.z = 3.0  # attribut non déclaré → AttributeError
except AttributeError as exc:
    print(exc)


---

## 6. `frozen=True` : immuabilité

Une dataclass frozen est **immuable** : toute tentative d'assignation lève `FrozenInstanceError`. Elle devient aussi **hachable automatiquement**.

In [ ]:
@dataclass(frozen=True, slots=True)
class Point:
    x: float
    y: float


In [ ]:
p = Point(3.0, 4.0)


In [ ]:
from dataclasses import FrozenInstanceError
try:
    p.x = 5.0
except FrozenInstanceError as exc:
    print(exc)


In [ ]:
{Point(1, 2), Point(1, 2), Point(3, 4)}


---

## 7. `kw_only=True` : arguments nommés

Force l'utilisation de paramètres nommés. Améliore la lisibilité des appels et évite les erreurs d'ordre.

In [ ]:
@dataclass(kw_only=True)
class Reservation:
    salle: str
    creneau: str
    organisateur: str


In [ ]:
Reservation(salle='Mars', creneau='lundi 9h', organisateur='Alice')


In [ ]:
try:
    Reservation('Mars', 'lundi 9h', 'Alice')  # positionnel interdit
except TypeError as exc:
    print(exc)


---

## 8. `__post_init__` : validation légère

Exécuté **après** le `__init__` généré. Idéal pour valider ou calculer des valeurs dérivées.

In [ ]:
@dataclass
class Intervalle:
    debut: int
    fin: int

    def __post_init__(self) -> None:
        if self.debut >= self.fin:
            raise ValueError('debut doit être < fin')
        self.duree = self.fin - self.debut


In [ ]:
Intervalle(9, 17)


In [ ]:
try:
    Intervalle(10, 10)
except ValueError as exc:
    print(exc)


---

## 9. Héritage de dataclasses

On peut hériter, mais attention à l'ordre : les attributs **avec défaut** du parent doivent précéder les attributs **sans défaut** de l'enfant, sauf avec `kw_only=True` qui contourne le problème.

In [ ]:
@dataclass(kw_only=True)
class Entite:
    id: int
    cree_a: str = ''

@dataclass(kw_only=True)
class Utilisateur(Entite):
    nom: str
    email: str


In [ ]:
Utilisateur(id=1, nom='Alice', email='a@ex.fr')


---

## 10. Conversion en dict avec `asdict`

In [ ]:
from dataclasses import asdict

@dataclass
class Produit:
    nom: str
    prix: float
    tags: list[str] = field(default_factory=list)

p = Produit('pain', 1.2, ['alim', 'base'])
asdict(p)


---

## 11. Limites et quand passer à Pydantic

`@dataclass` est parfait pour un **objet de données simple**. Ses limites :

- **pas de validation** au-delà d'une condition Python dans `__post_init__` ;
- **pas de coercition** automatique (str → int) ;
- **pas de sérialisation JSON** prête à l'emploi avec format contrôlé ;
- **pas de gestion fine des alias** de champs.

Dès que vous avez besoin d'une de ces briques, passez à **Pydantic v2** (notebook suivant).

---

## Synthèse

| Option | Rôle |
|---|---|
| `@dataclass` | Génère `__init__`, `__repr__`, `__eq__` |
| `field(default_factory=...)` | Valeur par défaut mutable |
| `slots=True` | `__slots__` auto, gain mémoire |
| `frozen=True` | Immuable, hachable |
| `kw_only=True` | Arguments nommés obligatoires |
| `__post_init__` | Validation / dérivation |
| `asdict(obj)` | Conversion en dict |


### Règles à retenir

1. **`@dataclass(slots=True)` par défaut** sur toute classe de données.
2. **Jamais `= []` en défaut** : toujours `field(default_factory=list)`.
3. **`frozen=True` quand l'objet représente une valeur** (Point, Coord, Vector…).
4. **`kw_only=True` dès 3 attributs**, pour la lisibilité des appels.
5. **Validation poussée = Pydantic**, pas `__post_init__` qui explose.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Dataclass minimale *(facile)*

Définir une dataclass `Livre` avec `titre: str`, `auteur: str`, `pages: int`. Créer une instance et la comparer à une autre identique.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Dataclasses", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass

@dataclass
class Livre:
    titre: str
    auteur: str
    pages: int

a = Livre('1984', 'Orwell', 328)
b = Livre('1984', 'Orwell', 328)
print(a == b)
```

</details>

### Exercice 2 — Liste par défaut *(facile)*

Définir `Equipe` avec `nom: str` et `membres: list[str]` initialisé à une liste vide. Ajouter deux membres puis afficher l'équipe.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Dataclasses", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass, field

@dataclass
class Equipe:
    nom: str
    membres: list[str] = field(default_factory=list)

e = Equipe('Core')
e.membres.extend(['Alice', 'Bob'])
print(e)
```

</details>

### Exercice 3 — Frozen + hash *(moyen)*

Définir `Couleur` (frozen + slots) avec `r, g, b` en `int`. Vérifier qu'on peut la mettre dans un `set`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Dataclasses", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Couleur:
    r: int
    g: int
    b: int

palette = {Couleur(255, 0, 0), Couleur(0, 255, 0), Couleur(255, 0, 0)}
print(palette)
```

</details>

### Exercice 4 — Validation dans `__post_init__` *(moyen)*

Définir `Utilisateur` avec `nom`, `email`, `age`. Dans `__post_init__`, vérifier que `age >= 0`, que l'email contient `'@'` et que `nom` n'est pas vide.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Dataclasses", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from dataclasses import dataclass

@dataclass
class Utilisateur:
    nom: str
    email: str
    age: int

    def __post_init__(self) -> None:
        if not self.nom:
            raise ValueError('nom vide')
        if '@' not in self.email:
            raise ValueError('email invalide')
        if self.age < 0:
            raise ValueError('age négatif')

print(Utilisateur('Alice', 'a@ex.fr', 30))
for test in [('', 'a@b', 10), ('X', 'bademail', 10), ('Y', 'a@b', -1)]:
    try:
        Utilisateur(*test)
    except ValueError as exc:
        print('refusé :', exc)
```

</details>

---

## Ressources externes

### Documentation officielle
- [`dataclasses` — stdlib](https://docs.python.org/3/library/dataclasses.html)

### PEPs de référence
- **PEP 557** — Data Classes

### Lectures complémentaires
- Raymond Hettinger, *Dataclasses: The code generator to end all code generators* (PyCon 2018).